# MS2 Sections 1-2: Dataset Motivation and Data Wrangling
Canvas Project #: TODO

Group Members: TODO

## Scope of This Notebook
This notebook supports presentation sections 1 and 2 only:
1. Introduction, motivation, and why `GSE96583` is a defensible dataset choice.
2. Data access, raw structure, wrangling, and preprocessing decisions.


## Data Description
`GSE96583` is a PBMC single-cell RNA-seq dataset from GEO with directly downloadable raw count matrices and cell-level metadata. It is a strong MS2 dataset because it already contains usable cell-type annotations, clear batch structure (`batch1` vs `batch2`), and a condition shift inside `batch2` (`ctrl` vs `stim`).

This makes it suitable for studying distribution shift without introducing a second dataset whose labels or accessibility are uncertain.


In [ ]:
from pathlib import Path
import pandas as pd

raw_dir = Path('/home/zichende/scRNA-ssl-uncertainty/data/raw/GSE96583')
files = []
for path in sorted(raw_dir.glob('*')):
    if path.is_file():
        files.append({'file': path.name, 'size_mb': round(path.stat().st_size / (1024 * 1024), 2)})
pd.DataFrame(files)


## Understanding the Raw Data Structure
The raw data are not a single clean table. Instead, the dataset is split across multiple compressed files:
- count matrices for different biological or technical subsets
- gene metadata files
- cell-level metadata files with t-SNE coordinates and annotations

That raw structure is one reason wrangling matters here. Before we can model anything, we have to unify matrices, reconcile gene spaces, attach metadata, and decide what cells to keep.


In [ ]:
import anndata as ad

adata_b1 = ad.read_h5ad('/home/zichende/scRNA-ssl-uncertainty/data/processed/GSE96583_batch1_qc_shared_annotated_singlets.h5ad')
adata_b2 = ad.read_h5ad('/home/zichende/scRNA-ssl-uncertainty/data/processed/GSE96583_batch2_qc_shared_annotated_singlets.h5ad')

print('batch1 shape:', adata_b1.shape)
print('batch2 shape:', adata_b2.shape)
print('batch1 samples:', adata_b1.obs['sample'].value_counts().to_dict())
print('batch2 samples:', adata_b2.obs['sample'].value_counts().to_dict())


## Wrangling and Preprocessing Decisions
The current processed benchmark reflects these choices:
- load and standardize the raw matrices from GEO
- attach available metadata to each cell
- restrict analyses to shared genes where cross-batch comparisons are valid
- apply QC-oriented filtering already captured in the processed `.h5ad` files
- keep only singlets to avoid doublets contaminating downstream cell-type classification

These are not cosmetic steps. They directly determine whether cross-batch evaluation is meaningful.


In [ ]:
label_col_b1 = 'cell.type' if 'cell.type' in adata_b1.obs.columns else 'cell'
label_col_b2 = 'cell.type' if 'cell.type' in adata_b2.obs.columns else 'cell'

label_counts = pd.DataFrame({
    'batch1': adata_b1.obs[label_col_b1].value_counts(),
    'batch2': adata_b2.obs[label_col_b2].value_counts(),
}).fillna(0).astype(int)
label_counts


## Meaningful Insights from Sections 1 and 2
- `GSE96583` is accessible, labeled, and already contains both batch shift and condition shift, so it is enough for a clean milestone dataset.
- The dataset is class-imbalanced: broad immune populations dominate while dendritic cells and megakaryocytes are rare.
- The raw file layout is fragmented, so wrangling is necessary before any valid analysis.
- Singlet filtering is important because downstream classification assumes one biological cell state per observation.


## Revised Research Question
Given a well-wrangled and annotated `GSE96583` PBMC benchmark, can self-supervised representations improve cell-type prediction robustness under batch and condition shift, and can uncertainty estimates help identify unreliable predictions?
